# EVIDENCE-X — Scientific Discovery & Evidence Intelligence
### An advanced Bioinformatics + NLP + RAG portfolio project

**Goal:** turn a collection of scientific papers into a searchable evidence workspace that extracts biomedical entities, identifies candidate claims, retrieves related evidence, flags potentially opposing findings, and builds a knowledge graph.

> This notebook is a working prototype, not a validated scientific evidence-synthesis system. The included example abstracts are synthetic demonstration text. Do not use its outputs for clinical decisions or cite generated claims as real research findings.

**Workflow**
1. Load abstracts or extract text from PDFs.
2. Clean and segment the text.
3. Identify biomedical entities with transparent baseline rules.
4. Extract candidate evidence statements.
5. Search semantically with TF-IDF.
6. Flag *potential* agreement / disagreement for human review.
7. Build and visualize a knowledge graph.
8. Export a structured evidence table.

## 0. Environment setup

Run this cell once in Google Colab or a local Jupyter environment. For PDF extraction, this notebook uses `pypdf`. The baseline NLP pipeline uses lightweight libraries so the first run is accessible.

In [ ]:
%pip -q install pandas numpy scikit-learn matplotlib networkx pypdf

## 1. Imports and configuration

In [ ]:
import re
import json
import textwrap
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

RANDOM_STATE = 42
pd.set_option("display.max_colwidth", 180)


## 2. Load a starter corpus

Start with the synthetic examples below to test the full pipeline. Replace them with real abstracts only after checking their source, DOI/PMID, and licensing/usage conditions.

In [ ]:
papers = [
    {
        "paper_id": "DEMO001",
        "title": "Illustrative study of GeneA expression in inflammatory disease",
        "year": 2022,
        "source": "Synthetic demonstration",
        "doi": None,
        "abstract": (
            "In this synthetic example, GeneA expression was increased in the "
            "case group compared with controls. The authors report an association "
            "between GeneA expression and inflammatory activity. The study does "
            "not establish that GeneA causes the disease."
        )
    },
    {
        "paper_id": "DEMO002",
        "title": "Illustrative analysis of GeneA and inflammatory pathways",
        "year": 2023,
        "source": "Synthetic demonstration",
        "doi": None,
        "abstract": (
            "This synthetic follow-up reports no significant difference in GeneA "
            "expression between cases and controls. The authors suggest that "
            "sample size and tissue selection may explain the inconsistent result."
        )
    },
    {
        "paper_id": "DEMO003",
        "title": "Illustrative protein interaction study",
        "year": 2024,
        "source": "Synthetic demonstration",
        "doi": None,
        "abstract": (
            "In a synthetic cell-based experiment, ProteinB interacted with "
            "ProteinC and was associated with PathwayD activity. Further validation "
            "is required before inferring a direct mechanism in humans."
        )
    }
]

papers_df = pd.DataFrame(papers)
papers_df[["paper_id", "title", "year", "source"]]


### Optional: upload PDF papers in Google Colab

For local Jupyter, put PDFs in a folder and use the local-folder alternative below. PDF extraction can lose table layout, equations, references, and multi-column reading order; always inspect extracted text.

In [ ]:
# Google Colab upload option:
try:
    from google.colab import files
    uploaded = files.upload()
    print("Uploaded:", list(uploaded.keys()))
except Exception:
    print("Not running in Google Colab. Use the local-folder option below if needed.")


In [ ]:
# Local-folder PDF option (edit the folder path):
# PDF_DIR = Path("./papers")
# from pypdf import PdfReader
#
# pdf_records = []
# for pdf_path in PDF_DIR.glob("*.pdf"):
#     reader = PdfReader(str(pdf_path))
#     full_text = "\\n".join(page.extract_text() or "" for page in reader.pages)
#     pdf_records.append({
#         "paper_id": pdf_path.stem,
#         "title": pdf_path.stem,
#         "year": None,
#         "source": str(pdf_path),
#         "doi": None,
#         "abstract": full_text
#     })
# if pdf_records:
#     papers_df = pd.DataFrame(pdf_records)
#     print("Loaded", len(papers_df), "PDFs")


## 3. Text preprocessing

In [ ]:
def clean_text(text):
    text = str(text or "")
    text = re.sub(r"\\s+", " ", text)
    return text.strip()

papers_df["text"] = papers_df["abstract"].map(clean_text)
papers_df[["paper_id", "text"]]


## 4. Baseline biomedical entity extraction

This deliberately transparent baseline recognizes entities from a small configurable vocabulary. For serious use, replace or augment it with SciSpacy, PubTator, or a domain-specific biomedical NER model and validate against manually annotated examples.

In [ ]:
ENTITY_LEXICON = {
    "GENE": ["GeneA"],
    "PROTEIN": ["ProteinB", "ProteinC"],
    "DISEASE": ["inflammatory disease"],
    "PATHWAY": ["PathwayD", "inflammatory pathway", "inflammatory activity"]
}

def extract_entities(text, lexicon=ENTITY_LEXICON):
    found = []
    for entity_type, terms in lexicon.items():
        for term in terms:
            if re.search(rf"(?<!\\w){re.escape(term)}(?!\\w)", text, flags=re.I):
                found.append({"entity": term, "entity_type": entity_type})
    return found

papers_df["entities"] = papers_df["text"].apply(extract_entities)
papers_df[["paper_id", "entities"]]


## 5. Candidate evidence statement extraction

A simple sentence-level heuristic selects sentences containing biomedical entities and evidence-related language. This is a starting point for a later transformer-based relation/claim extraction model, not a substitute for expert review.

In [ ]:
EVIDENCE_CUES = [
    "increased", "decreased", "higher", "lower", "associated",
    "interacted", "no significant difference", "correlated",
    "suggest", "reported", "observed", "causes", "causal"
]

def split_sentences(text):
    return [s.strip() for s in re.split(r"(?<=[.!?])\\s+", text) if s.strip()]

def extract_candidate_claims(row):
    claims = []
    for sentence in split_sentences(row["text"]):
        has_entity = len(extract_entities(sentence)) > 0
        has_cue = any(cue in sentence.lower() for cue in EVIDENCE_CUES)
        if has_entity and has_cue:
            claims.append(sentence)
    return claims

papers_df["candidate_claims"] = papers_df.apply(extract_candidate_claims, axis=1)
claim_rows = []
for _, row in papers_df.iterrows():
    for idx, claim in enumerate(row["candidate_claims"], start=1):
        claim_rows.append({
            "claim_id": f'{row["paper_id"]}_C{idx}',
            "paper_id": row["paper_id"],
            "title": row["title"],
            "year": row["year"],
            "claim": claim,
            "entities": extract_entities(claim)
        })

claims_df = pd.DataFrame(claim_rows)
claims_df


## 6. Evidence search with TF-IDF

Search retrieves textually similar candidate claims. TF-IDF is a baseline lexical method; it is not a full semantic embedding model. A later version can use sentence-transformers and a vector database.

In [ ]:
if len(claims_df):
    vectorizer = TfidfVectorizer(ngram_range=(1, 2), stop_words="english")
    claim_matrix = vectorizer.fit_transform(claims_df["claim"].tolist())

    def search_evidence(query, top_k=5):
        query_vec = vectorizer.transform([query])
        scores = cosine_similarity(query_vec, claim_matrix).ravel()
        order = np.argsort(scores)[::-1][:top_k]
        results = claims_df.iloc[order].copy()
        results["similarity"] = scores[order]
        return results[results["similarity"] > 0].reset_index(drop=True)
else:
    def search_evidence(query, top_k=5):
        return pd.DataFrame(columns=["claim_id", "paper_id", "claim", "similarity"])

search_evidence("GeneA expression in inflammatory disease", top_k=5)


## 7. Potentially opposing evidence — review queue

The rule below flags claims that mention the same entity but contain opposing cue patterns (for example, “increased” versus “no significant difference”). This is only a *candidate contradiction detector*: different populations, methods, tissues, endpoints, and statistical power can explain apparently different results. Human review is mandatory.

In [ ]:
POSITIVE_CUES = ["increased", "higher", "elevated", "upregulated"]
NEGATIVE_CUES = ["decreased", "lower", "reduced", "downregulated", "no significant difference", "not significant"]

def polarity_hint(claim):
    t = claim.lower()
    pos = any(x in t for x in POSITIVE_CUES)
    neg = any(x in t for x in NEGATIVE_CUES)
    if pos and not neg:
        return "positive-direction"
    if neg and not pos:
        return "negative-or-null"
    return "unclear"

claims_df["polarity_hint"] = claims_df["claim"].apply(polarity_hint)

review_pairs = []
for i in range(len(claims_df)):
    for j in range(i + 1, len(claims_df)):
        a, b = claims_df.iloc[i], claims_df.iloc[j]
        ents_a = {e["entity"].lower() for e in a["entities"]}
        ents_b = {e["entity"].lower() for e in b["entities"]}
        shared = sorted(ents_a & ents_b)
        opposing = {a["polarity_hint"], b["polarity_hint"]} == {
            "positive-direction", "negative-or-null"
        }
        if shared and opposing:
            review_pairs.append({
                "claim_a": a["claim_id"], "paper_a": a["paper_id"], "text_a": a["claim"],
                "claim_b": b["claim_id"], "paper_b": b["paper_id"], "text_b": b["claim"],
                "shared_entities": ", ".join(shared),
                "review_status": "Needs human review"
            })

review_df = pd.DataFrame(review_pairs)
review_df


## 8. Build a scientific knowledge graph

Nodes represent papers and extracted entities. Edges indicate that a paper mentions an entity. These edges mean *co-mention*, not causation, regulation, or experimentally validated interaction.

In [ ]:
G = nx.Graph()

for _, row in papers_df.iterrows():
    paper_node = f'PAPER:{row["paper_id"]}'
    G.add_node(paper_node, node_type="paper", label=row["paper_id"])
    for ent in row["entities"]:
        entity_node = f'{ent["entity_type"]}:{ent["entity"]}'
        G.add_node(entity_node, node_type=ent["entity_type"].lower(), label=ent["entity"])
        G.add_edge(paper_node, entity_node, relation="mentions")

print("Nodes:", G.number_of_nodes(), "| Edges:", G.number_of_edges())


In [ ]:
plt.figure(figsize=(11, 7))
pos = nx.spring_layout(G, seed=RANDOM_STATE, k=0.9)
node_colors = [
    "#4C78A8" if G.nodes[n].get("node_type") == "paper" else "#F2A541"
    for n in G.nodes
]
nx.draw_networkx(
    G, pos, with_labels=True, node_color=node_colors,
    node_size=1800, font_size=8, edge_color="#999999"
)
plt.title("EVIDENCE-X: Paper–Entity Co-mention Graph")
plt.axis("off")
plt.show()


## 9. Evidence dashboard

In [ ]:
print(f"Papers loaded: {len(papers_df)}")
print(f"Candidate claims: {len(claims_df)}")
print(f"Potential disagreement pairs for review: {len(review_df)}")
print(f"Knowledge graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

if not claims_df.empty:
    display(claims_df[["claim_id", "paper_id", "year", "claim", "polarity_hint"]])
if not review_df.empty:
    display(review_df)


## 10. Export results

The notebook exports candidate claims, review pairs, and paper metadata as CSV/JSON. Keep source metadata with every claim so a researcher can trace each statement back to its original document.

In [ ]:
OUTPUT_DIR = Path("evidence_x_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

papers_df.drop(columns=["entities", "candidate_claims"], errors="ignore").to_csv(
    OUTPUT_DIR / "papers.csv", index=False
)
claims_export = claims_df.copy()
if "entities" in claims_export.columns:
    claims_export["entities"] = claims_export["entities"].apply(json.dumps)
claims_export.to_csv(OUTPUT_DIR / "candidate_claims.csv", index=False)
review_df.to_csv(OUTPUT_DIR / "potential_disagreements_for_review.csv", index=False)

with open(OUTPUT_DIR / "knowledge_graph.json", "w", encoding="utf-8") as f:
    json.dump(nx.node_link_data(G), f, ensure_ascii=False, indent=2)

print("Saved outputs to:", OUTPUT_DIR.resolve())


## 11. Roadmap: turn the prototype into an industry-grade system

**Phase 1 — Data ingestion**
- Add PubMed / Europe PMC metadata retrieval with stable PMID/DOI provenance.
- Store PDF source, extraction timestamp, page number, and paragraph location.
- Add deduplication by DOI, PMID, and normalized title.

**Phase 2 — Biomedical NLP**
- Replace the demo lexicon with biomedical NER.
- Extract subject–predicate–object relations and negation.
- Preserve evidence spans and confidence scores.
- Evaluate on a manually annotated gold set.

**Phase 3 — RAG and evidence synthesis**
- Add sentence-transformer embeddings and FAISS/Chroma.
- Retrieve claims with citations and source snippets.
- Require every generated summary statement to link to evidence.
- Separate supporting, opposing, and insufficient evidence.

**Phase 4 — Knowledge graph**
- Move from co-mentions to typed, provenance-backed relations.
- Add Neo4j, entity normalization, and ontology identifiers.
- Never interpret co-mention as biological interaction.

**Phase 5 — Evaluation and deployment**
- Measure NER precision/recall/F1.
- Measure retrieval Precision@K / Recall@K / MRR.
- Manually assess contradiction-detection precision.
- Add FastAPI, React, authentication, tests, Docker, and cloud deployment.

### Suggested GitHub structure
```text
evidence-x/
├── notebooks/
│   └── EVIDENCE_X_Prototype.ipynb
├── data/
│   ├── raw/                 # keep private / respect licenses
│   └── processed/
├── src/
│   ├── ingestion/
│   ├── preprocessing/
│   ├── nlp/
│   ├── retrieval/
│   ├── evidence/
│   └── graph/
├── tests/
├── app/
├── outputs/
├── requirements.txt
└── README.md
```

**Important scientific limitation:** a system can surface candidate evidence and possible inconsistencies, but it cannot establish biological truth from text patterns alone. Validate every important result against the original paper and domain expertise.